In [20]:
import matplotlib.pyplot as plt
import utils_, config, model
import os
import numpy as np
import pandas as pd
import Analysis_function
import preprocess

import warnings
warnings.filterwarnings('ignore')


# 1. Fetch dataset in dict format - Work data / Head data
dataset_Work = preprocess.get_all_data(config.year_list, config.file_names_Work)
dataset_Head = preprocess.get_all_data(config.year_list, config.file_names_Head)

# 2. Check the every column and its semantic name
#utils_.see_col_idx_and_name(dataset['2020']['data'], dataset['2020']['meta'])

# 3. Check the intersection for the number of organization - year by year (e.g. compare 2020 - 2021)
#_ = Analysis_function.compare_company_ids_in_dataset(dataset_Work)

# 4. Check the intersection for the number of organization - All years (2020-2023) - we do this again in next step
#_ = Analysis_function.get_common_company_ids_all_years(dataset_Work)

# 5. Check the intersection for the number of organization (All) and filter; select only the organization that involves throughout all years
dataset_Work = preprocess.filter_dataset_by_common_ids(dataset_Work)

# 6. Target variable check - 1. Type count (value_counts()) / 2. check Nan
Analysis_function.target_variable_check(dataset_Work, target_variable=config.target_col)

# 7. Nan 값 25% 언더면 정수형 평균값으로 넣고, 위면 해당 컬럼 삭제
dataset_Work = preprocess.clean_all_years(dataset_Work, columns_to_drop=[], verbose=True)

# 8. Store only the common columns in each yearly dataset - 컬럼이 다르면 안되니깐.
dataset_Work = preprocess.unify_columns_by_base_name(dataset_Work)


HCCP_2ndWave_Work_1st_v2.sav ===> 2020 data
HCCP_2ndWave_Work_2nd_v2.sav ===> 2021 data
HCCP_2ndWave_Work_3rd_v3.sav ===> 2022 data
HCCP_2ndWave_Work_4th.sav ===> 2023 data
HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data
공통 기업 ID 개수: 384
2020 필터링 후 행 개수: 7054
2021 필터링 후 행 개수: 7613
2022 필터링 후 행 개수: 7338
2023 필터링 후 행 개수: 8740
2020 - W20Q09A : NaN 0개 / 전체 7054개 (0.00%)
W20Q09A
3.0    2372
8.0    1682
4.0    1534
2.0     873
5.0     434
1.0     159
Name: count, dtype: int64
2021 - W21Q09A : NaN 0개 / 전체 7613개 (0.00%)
W21Q09A
-8.0    2331
 3.0    2318
 4.0    1511
 2.0     907
 5.0     369
 1.0     177
Name: count, dtype: int64
2022 - W22Q09A : NaN 0개 / 전체 7338개 (0.00%)
W22Q09A
3.0    2263
8.0    1922
4.0    1436
2.0     893
5.0     501
1.0     323
Name: count, dtype: int64
2023 - W23Q09A : NaN 0개 / 전체 8740개 (0.00%)
W23Q09A
3.0    4008
4.0    2034
2.0    1133
5.0   

In [21]:

for year in config.year_list:
    print(dataset_Work[year]['data'].shape)

(7054, 126)
(7613, 126)
(7338, 126)
(8740, 126)


In [23]:
for year in ['2020', '2021', '2022', '2023']:
    if year == '2021' or '2023':
        if year == '2021':
            # Merge data - Work (X) + Head (y)
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')

            # Clean y data - i) erase labels 3/8/-8 and transform 1,2 -> 0 / 4,5 -> 1    ii) remove y from X
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            X['model20_21_output'] = model_20_21.predict_proba(X)[:, 1]
            #df_21_21['model20_21_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0) # 예측 확률이 0.6 이상인 경우만 feature로 사용, 나머지는 0으로
            acc = model.train_evaluate_model(X, y, learning_graph_show=False)

            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            X['model20_21_output'] = model_20_23.predict_proba(X)[:, 1]
            acc2 = model.train_evaluate_model(X, y, learning_graph_show=False)

        elif year == '2023':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc = model.train_evaluate_model(X, y, learning_graph_show=False)

    if year == '2020' or '2022':
        if year == '2020':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')

            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, model_20_21 = model.train_evaluate_model(X, y, learning_graph_show=False)

            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            acc2, model_20_23 = model.train_evaluate_model(X, y, learning_graph_show=False)

        elif year == '2022':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc = model.train_evaluate_model(X, y, learning_graph_show=False)


Year: 2020 | Merged shape: (7054, 124) | work shape: (7054, 126) | head shape: (500, 2)
Year: 2020 | Merged shape: (7054, 124) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1247
0.0     671
Name: count, dtype: int64
Resampled class distribution:
0.0    955
1.0    955
Name: count, dtype: int64
XGBoost Accuracy ========>  84.11458333333334 %

🧪 Processing: 
Original class distribution:
1.0    1388
0.0     309
Name: count, dtype: int64
Resampled class distribution:
1.0    1086
0.0    1086
Name: count, dtype: int64
XGBoost Accuracy ========>  92.64705882352942 %
Year: 2021 | Merged shape: (7613, 124) | work shape: (7613, 126) | head shape: (500, 2)
Year: 2021 | Merged shape: (7613, 124) | work shape: (7613, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1280
0.0     709
Name: count, dtype: int64
Resampled class distribution:
1.0    974
0.0    974
Name: count, dtype: int64
XGBoost Accuracy ========>  85.